# Gemma-3 270M fine-tune (QLoRA + LoRA)

Cleaned notebook that converts your NER-style HF dataset into SFT examples and shows a practical QLoRA + LoRA recipe suitable for Kaggle T4 GPUs.

Run cells in order.

In [ ]:
# Install required packages (run once). On Kaggle you may need to restart kernel after this cell.
!pip install -q transformers accelerate bitsandbytes peft datasets safetensors trl huggingface_hub

In [ ]:
# Standard imports
import os, json
from datasets import load_dataset, DatasetDict
import torch
from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTTrainingArguments

print('torch:', torch.__version__)

In [ ]:
# Optionally set HF token here (recommended to use Kaggle secret)
# os.environ['HUGGINGFACE_HUB_TOKEN'] = 'your_token_here'
from huggingface_hub import whoami
try:
    print('HF whoami:', whoami())
except Exception:
    print('Not logged in to HF - set HUGGINGFACE_HUB_TOKEN if you want to push artifacts')

In [ ]:
# 1) Load the dataset from Hugging Face (your dataset: NHLOCAL/SingNER)
raw = load_dataset('NHLOCAL/SingNER')
print(raw)
# print one example to inspect
print('Example:', raw['train'][0])

In [ ]:
# 2) Convert NER-format (start/end/label) into SFT instruction-response pairs.
def ner_to_sft(example):
    text = example.get('text', '')
    singers = []
    albums = []
    ents = example.get('entities', []) or example.get('labels', []) or []
    # entities expected as list of dicts with start,end,label
    for e in ents:
        start = e.get('start') if isinstance(e, dict) else None
        end = e.get('end') if isinstance(e, dict) else None
        label = e.get('label') if isinstance(e, dict) else None
        if start is None or end is None:
            continue
        name = text[start:end]
        if label == 'SINGER':
            singers.append(name.strip())
        elif label == 'ALBUM':
            albums.append(name.strip())
    # dedupe preserving order
    def dedupe(seq):
        seen = set(); out = []
        for x in seq:
            if x and x not in seen:
                seen.add(x); out.append(x)
        return out
    singers = dedupe(singers)
    albums = dedupe(albums)
    instruction = 'Extract singer and album names from the following text. Return a JSON object with keys "singers" and "albums".\n\nText:\n'
    prompt = instruction + text + '\n\nAnswer:'
    response = json.dumps({'singers': singers, 'albums': albums}, ensure_ascii=False)
    return {'input': prompt, 'output': response}

# Apply conversion to all splits
sft_splits = {}
for split in raw.keys():
    print('Converting split:', split)
    sft_splits[split] = raw[split].map(lambda ex: ner_to_sft(ex), remove_columns=raw[split].column_names)

sft = DatasetDict(sft_splits)
print(sft)

In [ ]:
# Quick validation of converted examples
for i in range(min(3, len(sft['train']))):
    print('---')
    print('INPUT:', sft['train'][i]['input'][:400])
    print('OUTPUT:', sft['train'][i]['output'])

In [ ]:
# Save converted dataset locally (optional)
os.makedirs('data', exist_ok=True)
sft['train'].to_json('data/train_sft.jsonl')
if 'test' in sft:
    sft['test'].to_json('data/valid_sft.jsonl')
print('Saved converted dataset to data/*.jsonl')

In [ ]:
# Tokenizer and model id
model_name = 'google/gemma-3-270m'
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})
print('Tokenizer loaded. Vocab size:', len(tokenizer))

In [ ]:
# Prepare 4-bit loading config (bitsandbytes / QLoRA style)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4'
)
print('BitsAndBytes config created.\nNote: model download may be large and require internet access.')

In [ ]:
# Load model in 4-bit and prepare for k-bit training (this may require a restart depending on environment)
config = AutoConfig.from_pretrained(model_name)
# The following line can be memory intensive; run on Kaggle with appropriate GPU
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map='auto',
    quantization_config=bnb_config,
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)
print('Model loaded and prepared for k-bit training.')

In [ ]:
# Attach LoRA (PEFT)
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=['q_proj','v_proj','k_proj','o_proj','gate_proj','down_proj','up_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)
model = get_peft_model(model, lora_config)
print('LoRA adapters attached to the model.')

In [ ]:
# Tokenize function for SFT (concatenate prompt + response)
max_length = 512

def tokenize_fn(example):
    full = example['input'] + ' ' + example['output']
    toks = tokenizer(full, truncation=True, max_length=max_length, padding='max_length')
    return {'input_ids': toks['input_ids'], 'attention_mask': toks['attention_mask']}

train_tok = sft['train'].map(tokenize_fn, remove_columns=sft['train'].column_names)
valid_tok = sft['test'].map(tokenize_fn, remove_columns=sft['test'].column_names) if 'test' in sft else None
print('Tokenized dataset: train size =', len(train_tok))

In [ ]:
# SFT training args (small defaults for Kaggle T4 - adjust as needed)
training_args = SFTTrainingArguments(
    output_dir='outputs/gemma3-lora',
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_strategy='epoch',
    remove_unused_columns=False,
    save_total_limit=2,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=valid_tok if valid_tok is not None else None,
    tokenizer=tokenizer,
)

print('Trainer prepared. To start training run: trainer.train()')

In [ ]:
# NOTE: Running training here will start the fine-tuning loop and consume GPU time.
# trainer.train()
# trainer.save_model('outputs/gemma3-lora-final')
print('Training cell is commented out by default. Uncomment to run training.')

### Notes
- This notebook uses QLoRA (bitsandbytes 4-bit nf4) + LoRA adapters, which is the practical approach for small GPUs like T4.
- True QAT -> int4 workflows are research-level and require additional steps.
- Adjust batch sizes, gradient accumulation and epochs to fit your GPU memory and dataset size.

If you want, I can: (A) change the output format, (B) add an HF Hub push cell, or (C) provide a QAT→int4 research recipe.